In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Initialize Spark with Kafka package (adjust package version to match your Spark version if needed)
spark = SparkSession.builder \
    .appName("BigData_Lab2") \
    .config(
        "spark.jars.packages", 
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0"
    ) \
    .master("local[*]") \
    .getOrCreate()

# --- 1. DEFINE SCHEMAS ---
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

# --- 2. INGEST FROM KAFKA & APPLY SCHEMAS ---

# 2a. Movies Stream (Topic: Lab1_movies)
df_movies_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "Lab1_movies") \
    .load()

# Convert binary 'value' to string, parse JSON, and expand columns
df_movies = df_movies_raw.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), movies_schema).alias("data")) \
    .select("data.*")

# 2b. Ratings Stream (Topic: Lab1_ratings) with 100 rows/trigger limit
df_ratings_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "Lab1_ratings") \
    .option("maxOffsetsPerTrigger", 100) \
    .load()

df_ratings = df_ratings_raw.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), ratings_schema).alias("data")) \
    .select("data.*")

# 2c. Tags Stream (Topic: Lab1_tags)
df_tags_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "Lab1_tags") \
    .option("maxOffsetsPerTrigger", 100) \
    .load()

df_tags = df_tags_raw.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), tags_schema).alias("data")) \
    .select("data.*")

print("Starting the Ratings stream to console...")

# Define the sink (console) and start the stream
query_ratings = df_ratings.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()
query_tags = df_tags.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

# This is the crucial part: it keeps the Python script running forever 
# to continuously listen for new Kafka messages.
query_ratings.awaitTermination()
query_tags.awaitTermination()

Starting the Ratings stream to console...


26/04/28 19:28:03 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-610d1256-12b2-411f-9a99-50ddb9a15925. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/28 19:28:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/28 19:28:03 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-4f769240-e578-4641-85a2-e66c4f2c6e71. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/28 19:28:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not support

-------------------------------------------
Batch: 0
-------------------------------------------
-------------------------------------------
Batch: 0
-------------------------------------------
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
+------+-------+------+---------+

+------+-------+---+---------+
|userId|movieId|tag|timestamp|
+------+-------+---+---------+
+------+-------+---+---------+



-------------------------------------------
Batch: 328
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
-------------------------------------------
Batch: 328
-------------------------------------------
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|NULL  |NULL   |4.0   |NULL     |
+------+-------+------+---------+

+------+-------+-----------+----------+
|userId|movieId|tag        |timestamp |
+------+-------+-----------+----------+
|474   |3181   |Shakespeare|1137179352|
+------+-------+-----------+----------+

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|NULL  |NULL   |4.0   |NULL     |
+------+-------+------+---------+

+------+-------+-----------+----------+
|userId|movieId|tag      

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/liemak363/repo/Big Data Lab/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/liemak363/repo/Big Data Lab/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 